In [ ]:
"""
Kaggle Mitsui: streaming inference server with TimeXer + LoRA checkpoint.
Server-first layout: start server ASAP; lazy load model on first predict().
"""

import os
from collections import deque
import polars as pl
import time
import traceback
import numpy as np


import kaggle_evaluation.mitsui_inference_server

# <<< EDIT >>> attached dataset roots
WEIGHTS_DIR = "/kaggle/input/aurora_borealis_v2/pytorch/default/1"   # contains best.pt
SRC_ROOT    = "/kaggle/input/src-models-patch-embed"                 # contains timexer.py, lora.py

NUM_TARGET_COLUMNS = 424
WINDOW = 160

# Minimal global state; no heavy imports yet
_state = {
    "ready": False,
    "device": "cpu",    # set later
    "model": None,
    "expected_feats": None,
    "mu": None,
    "sig": None,
    "buf": deque(maxlen=WINDOW),
}

def _load_module(mod_name: str, file_path: str):
    import importlib.util, os
    assert os.path.exists(file_path), f"Missing file: {file_path}"
    spec = importlib.util.spec_from_file_location(mod_name, file_path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

def _robust_mu_sigma_train(train_df, feature_cols):
    import numpy as np
    X = train_df[feature_cols]
    mu  = X.median().to_numpy(np.float32)
    iqr = (X.quantile(0.75) - X.quantile(0.25)).to_numpy(np.float32)
    std  = X.std(ddof=0).to_numpy(np.float32)
    sig  = np.where(iqr > 1e-6, iqr, std)
    sig  = np.where(sig > 1e-6, sig, 1.0).astype(np.float32)
    return mu, sig

def _bootstrap_once(base_path: str):
    if _state["ready"]:
        return

    # Heavy imports here (after server has started)
    import numpy as np
    import pandas as pd
    import torch
    import glob

    # Load local code by file path (avoids sys.path issues)
    timexer_mod = _load_module("timexer", os.path.join(SRC_ROOT, "timexer.py"))
    lora_mod    = _load_module("lora",    os.path.join(SRC_ROOT, "lora.py"))
    build_timexer      = getattr(timexer_mod, "build_timexer")
    apply_lora_to_attn = getattr(lora_mod, "apply_lora_to_attn")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    _state["device"] = device

    # Find checkpoint robustly
    candidates = glob.glob(os.path.join(WEIGHTS_DIR, "**", "best.pt"), recursive=True)
    if not candidates:
        candidates = glob.glob(os.path.join(WEIGHTS_DIR, "**", "*.pt"), recursive=True)
    assert candidates, f"No .pt checkpoint found under {WEIGHTS_DIR}"
    ckpt_path = candidates[0]
    ckpt = torch.load(ckpt_path, map_location=device)

    # Train-only stats (first predict call is exempt from 1-min limit)
    train_pl = pl.read_csv(os.path.join(base_path, "train.csv"))
    all_train_cols = [c for c in train_pl.columns if c != "date_id" and not c.startswith("target_")]
    expected_feats = ckpt.get("feature_cols", all_train_cols)
    train_pd = train_pl.to_pandas()
    for c in expected_feats:
        if c not in train_pd.columns:
            train_pd[c] = 0.0
    mu, sig = _robust_mu_sigma_train(train_pd, expected_feats)

    # Build model and load weights
    model = build_timexer(
        num_features=len(expected_feats),
        num_targets=NUM_TARGET_COLUMNS,
        d_model=512, n_layers=6, n_heads=8, ffn_hidden=2048,
        dropout=0.1, patch_size=20, stride=20, use_cls=False
    )
    apply_lora_to_attn(model, r=8, alpha=16, freeze_base=True)
    model.load_state_dict(ckpt["model"], strict=True)
    model.eval().to(device)
    torch.set_grad_enabled(False)

    _state.update({
        "ready": True,
        "model": model,
        "expected_feats": expected_feats,
        "mu": mu,
        "sig": sig,
        "buf": deque(maxlen=WINDOW),
    })

def _row_to_feature_vector(row_pl: pl.DataFrame, expected_feats):
    import numpy as np
    cols_present = set(row_pl.columns)
    vals = []
    for c in expected_feats:
        if c in cols_present:
            v = row_pl.select(c).to_numpy().reshape(-1)
            vals.append(np.float32(v[0]))
        else:
            vals.append(np.float32(0.0))
    return np.asarray(vals, dtype=np.float32)

def _normalize_clip(x, mu, sig):
    import numpy as np
    x = (x - mu) / sig
    np.clip(x, -8.0, 8.0, out=x)
    return x

def _collect_row_values(frames: list[pl.DataFrame]) -> dict[str, float]:
    """Take several 1-row Polars frames and return a {col: value} map.
    First occurrence wins; later frames with same col name are ignored."""
    vals = {}
    for f in frames:
        for c in f.columns:
            if c in vals:
                continue
            v = f.select(c).to_numpy().reshape(-1)
            vals[c] = float(np.float32(v[0])) if len(v) else 0.0
    return vals


DEBUG_LOG = "/kaggle/working/gateway_debug.log"

def predict(test: pl.DataFrame,
            label_lags_1_batch: pl.DataFrame,
            label_lags_2_batch: pl.DataFrame,
            label_lags_3_batch: pl.DataFrame,
            label_lags_4_batch: pl.DataFrame):

    t0 = time.time()
    base_path = "/kaggle/input/mitsui-commodity-prediction-challenge"
    try:
        _bootstrap_once(base_path)

        model = _state["model"]
        expected_feats = _state["expected_feats"]
        mu, sig = _state["mu"], _state["sig"]
        buf = _state["buf"]
        device = _state["device"]

        vals_map = _collect_row_values([
            test, label_lags_1_batch, label_lags_2_batch, label_lags_3_batch, label_lags_4_batch
        ])

        # Build feature vector in the exact expected order
        x = np.asarray([np.float32(vals_map.get(c, 0.0)) for c in expected_feats], dtype=np.float32)
        x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
        x = _normalize_clip(x, mu, sig)

        buf.append(x)
        if len(buf) < WINDOW:
            c = len(expected_feats)
            pad = [np.zeros(c, dtype=np.float32)] * (WINDOW - len(buf))
            seq = np.stack(list(pad) + list(buf), axis=0)
        else:
            seq = np.stack(buf, axis=0)

        import torch
        xb = torch.from_numpy(seq).unsqueeze(0).to(device)   # (1, T, C)
        with torch.no_grad():
            pb = model(xb).detach().cpu().numpy().reshape(-1)

        # ---- pandas return, fixed order & dtype ----
        import pandas as pd
        cols = [f"target_{i}" for i in range(NUM_TARGET_COLUMNS)]
        out = pd.DataFrame([pb.astype("float32")], columns=cols)
        assert out.shape == (1, NUM_TARGET_COLUMNS)
        return out

    except Exception as e:
        msg = f"[predict ERROR] {type(e).__name__}: {e}\n{traceback.format_exc()}\n"
        print(msg, flush=True)
        with open(DEBUG_LOG, "a") as f:
            f.write(msg)
        # Re-raise so the gateway shows it as a failure
        raise
    finally:
        print(f"[predict] elapsed={time.time()-t0:.2f}s", flush=True)

# ---- Start the server ASAP (no heavy work above this line) ----
inference_server = kaggle_evaluation.mitsui_inference_server.MitsuiInferenceServer(predict)

if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    inference_server.serve()
else:
    inference_server.run_local_gateway(("/kaggle/input/mitsui-commodity-prediction-challenge/",))
